# nb9 — Hướng B · E0: MLM oracle cho correction-by-selection (CPU/T4-light)

Thực thi `REPORT.md` §15.2 (E0) qua plan `.kilo/plans/1790300265078-telex-normalize-mlm-oracle.md` (quyết định 25/09): CorrAcc@TP test Run 2 = **67,94%** là chỉ số thấp nhất; nb4 Q4 ~77% TP-sai là thay nhầm **từ thật** → model detect tốt nhưng sinh nội dung kém. E0 đo **trần CorrAcc** nếu correction là **selection** (chọn từ candidate set) thay vì generation — chỉ E0 trong plan này; E1 (rerank thật) mở riêng nếu oracle đạt ngưỡng.

**Candidate set mỗi vị trí TP-detect-nhưng-sai-nội-dung** (thứ tự pre-registered): (1) `identity` — giữ output model (về nguyên tắc không thể hit trên vị trí sai-nội-dung vì tập này dựng bằng phép so `.lower()` — vẫn giữ đủ đầy theo plan); (2) `confusion` — đảo bảng lỗi `noise_model.json` của nb2 (nguồn **train-only**); (3) **MLM top-k** infilling `vinai/bartpho-syllable` gốc (cùng không gian tokenizer với task, không thêm dependency; probe kém → fallback `VietAI/vit5-base` span-infilling — quyết định tại chỗ, ghi report).

**Verdict PRE-REGISTERED**: projected CorrAcc oracle@10 ≥ CorrAcc của cùng bộ prediction **+10pp** (nền test Run 2 67,94% → ≥ ~77,9%) → mở E1 (plan riêng); dưới ngưỡng → dừng Hướng B, ghi REPORT (selection không có trần đủ lớn).

**Chống leakage (`DESIGN.md` §9)**: candidates chỉ từ nguồn công khai (MLM) + train (confusion); nhãn test chỉ dùng để **đo** hit, không tham gia dựng candidate.

**Input**: nb3 (`predictions_{val,test}_run2.jsonl`) · nb2 (`noise_model.json` + `syllable_table.json`) · nb1 (`test_aligned.jsonl` đối chiếu) · model MLM qua HF (Internet ON lần đầu download).
**Output**: `mlm_oracle_report.json` (oracle@k, phân bố candidate-hit theo nguồn confusion/MLM/identity, QA 10 mẫu).

Chạy được thuần CPU (phần MLM ~1–2k vị trí, ước lượng 20–40 phút trên CPU; GPU nếu có thì nhanh hơn — kết quả oracle không phụ thuộc hardware vì chỉ đọc top-k).

## Nội dung
0. Cấu hình + guard + ngưỡng pre-registered + tự dò input
1. Cell hàm dùng chung (copy nguyên vẹn từ nb5 — `align-v1`) + `evaluate_predictions`
2. Dựng tập vị trí TP-detect-nhưng-sai-nội-dung (test + val)
3. Candidate set: confusion (train-only) + MLM top-k + probe/fallback
4. Oracle CorrAcc@k + breakdown + verdict
5. QA + xuất file

In [ ]:
!pip uninstall -y torchao==0.10.0

## 0. Cấu hình & guard môi trường

Mọi tham số gom một chỗ (`DESIGN.md` §7). Ngưỡng verdict in ra **trước** khi chạy. Không cần peft (không LoRA — dùng model gốc). Guard `transformers<5` giữ pattern comment như nb7/nb8.

In [ ]:
import os
import json
import random
import datetime
import unicodedata
from pathlib import Path
import collections

try:
    import sentencepiece
except ImportError:
    print('Cài đặt sentencepiece...')
    os.system('pip install -q sentencepiece')
    os.system('pip uninstall -y torchao')

# Guard version: transformers v5 bug convert tokenizer sentencepiece (ViT5/T5-style) → KeyError: 0
# (nb7/nb8 đã chạy thành công với guard comment-out ngày 24–25/09 — giữ nguyên pattern)
import transformers
# from packaging.version import Version
# if Version(transformers.__version__).major >= 5:
#     print(f'transformers {transformers.__version__} (v5) — downgrade về <5...')
#     os.system('pip install -q "transformers<5"')
#     raise SystemExit(
#         'Đã downgrade transformers về <5. BÂY GIỜ: Restart Kernel '
#         '(Run → Restart & Clear Outputs) rồi Run All lại từ đầu — guard sẽ pass và chạy bình thường.'
#     )

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed

SEED = 42
set_seed(SEED)

# --- Cấu hình nb9 ---
MLM_MODEL_ID = 'vinai/bartpho-syllable'     # chính — cùng không gian tokenizer với task, không thêm dependency
MLM_FALLBACK_ID = 'VietAI/vit5-base'        # fallback span-infilling (T5-style) — chỉ khi probe kém, ghi report
TOP_K = 20
K_ORACLE = [1, 5, 10, 20]
MLM_BATCH_SIZE = 16
MAX_SOURCE_LEN = 256
PROBE_MIN_HITS = 2          # probe: >= 2/4 case hiển nhiên hit trong top-10 mới coi infill đạt (smoke — không phải benchmark)
PROBE_CASES = [             # (tokens với '<mask>', token kỳ vọng)
    (['học', '<mask>', 'đi', 'học'], 'sinh'),
    (['<mask>', 'sinh', 'đi', 'học'], 'học'),
    (['Tôi', 'đi', '<mask>', 'học'], 'trường'),
    (['anh', 'ấy', 'đã', '<mask>', 'nhà'], 'về'),
]

# --- Verdict PRE-REGISTERED (plan 25/09) ---
ORACLE_K_DECISION = 10      # oracle@10
ORACLE_MIN_GAIN = 0.10      # projected CorrAcc oracle@10 >= CorrAcc nền + 10pp (~77,9% nếu nền 67,94%)
BASE_CORRACC_TEST_REF = 0.6794   # nền REPORT §13 — chỉ đối chiếu in ra; verdict dùng CorrAcc tính lại cùng pipeline

REQUIRED_FILES = [
    'predictions_val_run2.jsonl',    # nb3
    'predictions_test_run2.jsonl',   # nb3
    'noise_model.json',              # nb2 — CONFUSION nguồn train-only
    'syllable_table.json',           # nb2 — bảng âm tiết
    'test_aligned.jsonl',            # nb1 — đối chiếu text test
]


def find_required_inputs():
    candidates = []
    kaggle = Path('/kaggle/input')
    if kaggle.is_dir():
        candidates += sorted(kaggle.rglob('*.jsonl')) + sorted(kaggle.rglob('*.json'))
    local = Path('./out')
    if local.is_dir():
        candidates += sorted(local.rglob('*.jsonl')) + sorted(local.rglob('*.json'))
    found = {}
    for p in candidates:
        if p.name in REQUIRED_FILES and p.name not in found:
            found[p.name] = p
    missing = [req for req in REQUIRED_FILES if req not in found]
    if missing:
        raise FileNotFoundError(
            'Thiếu input bắt buộc: ' + ', '.join(missing) +
            ' | nb9 cần Add Input: output nb3 (predictions_{val,test}_run2.jsonl), '
            'output nb2 (noise_model.json + syllable_table.json), output nb1 (test_aligned.jsonl). '
            'Local: đặt file vào ./out.'
        )
    return found


INPUT_FILES = find_required_inputs()

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('./out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.datetime.now().isoformat(timespec='seconds')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FP16 = DEVICE == 'cuda'

print('=== NGƯỠNG VERDICT PRE-REGISTERED (in trước khi chạy) ===')
print(f'  E0: projected CorrAcc oracle@{ORACLE_K_DECISION} >= CorrAcc nền + {ORACLE_MIN_GAIN:.0%} '
      f'(nền tham chiếu test Run 2 = {BASE_CORRACC_TEST_REF:.2%}) → đạt mới mở E1 (plan riêng)')
print()
print('=== TỰ DÒ INPUT HOÀN TẤT (CPU-friendly — không cần peft/LoRA) ===')
for k in REQUIRED_FILES:
    print(f'  {k:32s}: {INPUT_FILES[k]}')
print(f'Device: {DEVICE.upper()} | FP16={FP16} | MLM chính: {MLM_MODEL_ID} (fallback: {MLM_FALLBACK_ID})')

## 1. Cell hàm dùng chung — copy NGUYÊN VẸN từ nb5 (`align-v1`) + `evaluate_predictions`

Như nb5/nb6/nb7/nb8. `evaluate_predictions` cần để lấy `total_tp` / `correct_at_tp` — CorrAcc nền của verdict được **tính lại cùng pipeline** trên bộ prediction đầu vào, không dùng hằng tham chiếu.

In [ ]:
import re

SHARED_CELLS_VERSION = 'align-v1'

_WS_RE = re.compile(r'\s+')
_TOKEN_RE = re.compile(r'\w+|[^\w\s]+')
_HAS_WORD_RE = re.compile(r'\w')
_DIGIT_RE = re.compile(r'\d')

def nfc_normalize(s):
    return _WS_RE.sub(' ', unicodedata.normalize('NFC', s)).strip()

def canon_tokenize(s):
    """NFC + tách token: run chữ/số liền kề (\\w+) là 1 token; cụm dấu câu liền kề là 1 token riêng."""
    return _TOKEN_RE.findall(nfc_normalize(s))

def is_punct_token(tok):
    return not _HAS_WORD_RE.search(tok)

def is_word_token(tok):
    return not is_punct_token(tok) and not _DIGIT_RE.search(tok)

def levenshtein_opcodes(src, tgt):
    n, m = len(src), len(tgt)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        dp[i][0] = i
    for j in range(1, m + 1):
        dp[0][j] = j
    for i in range(1, n + 1):
        si = src[i - 1]
        prev, row = dp[i - 1], dp[i]
        for j in range(1, m + 1):
            best = prev[j - 1] + (0 if si == tgt[j - 1] else 1)
            if prev[j] + 1 < best:
                best = prev[j] + 1
            if row[j - 1] + 1 < best:
                best = row[j - 1] + 1
            row[j] = best
    ops = []
    def _push(tag, i1, i2, j1, j2):
        if ops and ops[-1][0] == tag and ops[-1][1] == i2 and ops[-1][3] == j2:
            ops[-1] = (tag, i1, ops[-1][2], j1, ops[-1][4])
        else:
            ops.append((tag, i1, i2, j1, j2))
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + (0 if src[i - 1] == tgt[j - 1] else 1):
            _push('equal' if src[i - 1] == tgt[j - 1] else 'replace', i - 1, i, j - 1, j)
            i, j = i - 1, j - 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            _push('delete', i - 1, i, j, j)
            i -= 1
        else:
            _push('insert', i, i, j - 1, j)
            j -= 1
    ops.reverse()
    return ops

def _make_block(src, tgt, span):
    i1, i2, j1, j2 = span
    src_toks, tgt_toks = src[i1:i2], tgt[j1:j2]
    ns, nt = len(src_toks), len(tgt_toks)
    if ns == 0:
        btype = 'insert'
    elif nt == 0:
        btype = 'delete'
    elif ns == 1 and nt == 1:
        btype = 'substitute'
    elif ns == 1:
        btype = 'split'
    elif nt == 1:
        btype = 'merge'
    else:
        btype = 'multi'
    return {
        'type': btype,
        'position': i1,
        'src_span': [i1, i2],
        'tgt_span': [j1, j2],
        'src_tokens': src_toks,
        'tgt_tokens': tgt_toks,
        'punct_only': all(is_punct_token(t) for t in src_toks + tgt_toks),
    }

def extract_edit_blocks(src, tgt, opcodes):
    blocks, cur = [], None
    for tag, i1, i2, j1, j2 in opcodes:
        if tag == 'equal':
            if cur is not None:
                blocks.append(_make_block(src, tgt, cur))
                cur = None
        elif cur is None:
            cur = [i1, i2, j1, j2]
        else:
            cur[1], cur[3] = i2, j2
    if cur is not None:
        blocks.append(_make_block(src, tgt, cur))
    return blocks

def apply_edit_blocks(src, blocks):
    out, pos = [], 0
    for b in blocks:
        out += src[pos:b['src_span'][0]]
        out += b['tgt_tokens']
        pos = b['src_span'][1]
    out += src[pos:]
    return out

SUSPECT_EDIT_RATIO = 0.3  # giữ nguyên giá trị nb1/nb2 — cell hàm dùng chung copy nguyên vẹn cần nó

def build_pseudo_annotation(text, corrected, suspect_ratio=SUSPECT_EDIT_RATIO):
    """Align text ↔ corrected_text trong hệ token canonical → pseudo-annotation schema giống VSEC + edit_blocks.
    Quy ước QUAN TRỌNG (các notebook sau phải dùng cell này nguyên vẹn):
    - error_positions: index token nguồn nằm trong src_span của block CÓ token nguồn.
      Block insert (thiếu âm tiết ở nguồn) KHÔNG có vị trí nguồn → không nằm trong error_positions,
      chỉ nằm trong correction_pairs với error='' và position = vị trí chèn trước (có thể == len(src)).
    - correction_pairs: 1 entry/block; error/correction = các token nối bằng space; delete → correction=''.
    - syllable_annotations: 1 entry/token nguồn; is_correct=False khi token thuộc src_span của block
      non-insert; corrections = chuỗi đích (join space) của block đó.
    - suspect: edit_ratio = (tổng token cả 2 vế nằm trong edit block) / max(len(src), len(tgt)) vượt ngưỡng.
    - align_failed: một trong hai vế token hóa rỗng."""
    src = canon_tokenize(text)
    tgt = canon_tokenize(corrected)
    blocks = extract_edit_blocks(src, tgt, levenshtein_opcodes(src, tgt))
    corrections_by_pos = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            fix = ' '.join(b['tgt_tokens'])
            for i in range(b['src_span'][0], b['src_span'][1]):
                corrections_by_pos.setdefault(i, []).append(fix)
    error_positions = sorted(corrections_by_pos)
    syllable_annotations = [
        {
            'syllable': tok,
            'is_correct': i not in corrections_by_pos,
            'corrections': corrections_by_pos.get(i, []),
            'position': i,
        }
        for i, tok in enumerate(src)
    ]
    correction_pairs = [
        {'error': ' '.join(b['src_tokens']), 'correction': ' '.join(b['tgt_tokens']), 'position': b['position']}
        for b in blocks
    ]
    n_edit_tokens = sum(
        (b['src_span'][1] - b['src_span'][0]) + (b['tgt_span'][1] - b['tgt_span'][0]) for b in blocks
    )
    denom = max(len(src), len(tgt))
    edit_ratio = n_edit_tokens / denom if denom else 0.0
    return {
        'is_clean': not blocks,
        'align_failed': not src or not tgt,
        'suspect': edit_ratio > suspect_ratio,
        'edit_ratio': round(edit_ratio, 4),
        'error_count': len(blocks),
        'error_positions': error_positions,
        'correction_pairs': correction_pairs,
        'syllable_annotations': syllable_annotations,
        'edit_blocks': blocks,
        'src_tokens': src,
        'tgt_tokens': tgt,
    }

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

print('SHARED_CELLS_VERSION:', SHARED_CELLS_VERSION)

### `evaluate_predictions` — copy nguyên vẹn từ nb5 (giống nb3 §1)

In [ ]:
def evaluate_predictions(records, predictions, syll_set=None):
    assert len(records) == len(predictions), f'Độ dài không khớp: {len(records)} vs {len(predictions)}'
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    correct_at_tp = 0
    
    total_clean_tokens = 0
    clean_sents_total = 0
    clean_sents_preserved = 0
    
    # Stratified stats: non-word vs real-word
    stratified = {
        'nonword': {'gold': 0, 'detected': 0, 'corrected': 0},
        'realword': {'gold': 0, 'detected': 0, 'corrected': 0}
    }
    
    sample_overcorrections = []
    
    for rec, pred in zip(records, predictions):
        src_toks = canon_tokenize(rec['text'])
        gold_toks = canon_tokenize(rec['corrected_text'])
        pred_toks = canon_tokenize(pred)
        
        # Gold edit blocks
        gold_ops = levenshtein_opcodes(src_toks, gold_toks)
        gold_blocks = extract_edit_blocks(src_toks, gold_toks, gold_ops)
        
        # Pred edit blocks
        pred_ops = levenshtein_opcodes(src_toks, pred_toks)
        pred_blocks = extract_edit_blocks(src_toks, pred_toks, pred_ops)
        
        gold_pos_map = {}
        for b in gold_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                target_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    gold_pos_map[p] = (target_str, b['src_tokens'])
                    
        pred_pos_map = {}
        for b in pred_blocks:
            if b['src_span'][0] < b['src_span'][1]:
                pred_str = ' '.join(b['tgt_tokens'])
                for p in range(b['src_span'][0], b['src_span'][1]):
                    pred_pos_map[p] = (pred_str, b['src_tokens'])
                    
        gold_positions = set(gold_pos_map.keys())
        pred_positions = set(pred_pos_map.keys())
        
        tp_pos = gold_positions & pred_positions
        fp_pos = pred_positions - gold_positions
        fn_pos = gold_positions - pred_positions
        
        total_tp += len(tp_pos)
        total_fp += len(fp_pos)
        total_fn += len(fn_pos)
        
        # Đếm số token đúng trong câu nguồn (loại bỏ token dấu câu)
        word_token_positions = {i for i, t in enumerate(src_toks) if is_word_token(t)}
        clean_word_positions = word_token_positions - gold_positions
        total_clean_tokens += len(clean_word_positions)
        
        # Check clean sentence
        if len(gold_positions) == 0:
            clean_sents_total += 1
            if len(pred_positions) == 0:
                clean_sents_preserved += 1
                
        # Correction accuracy
        for p in tp_pos:
            target_str, _ = gold_pos_map[p]
            pred_str, _ = pred_pos_map[p]
            if pred_str.lower() == target_str.lower():
                correct_at_tp += 1
                
            # Stratified
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['detected'] += 1
                if pred_str.lower() == target_str.lower():
                    stratified[cat]['corrected'] += 1
                    
        for p in fn_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
        for p in tp_pos:
            if syll_set is not None and p < len(src_toks):
                tok_lower = src_toks[p].lower()
                cat = 'nonword' if tok_lower not in syll_set else 'realword'
                stratified[cat]['gold'] += 1
                
        # Lưu mẫu over-correction tiêu biểu
        if fp_pos and len(sample_overcorrections) < 20:
            for p in sorted(fp_pos):
                if p < len(src_toks):
                    sample_overcorrections.append({
                        'orig_token': src_toks[p],
                        'pred_token': pred_pos_map[p][0],
                        'context': ' '.join(src_toks[max(0, p-3):min(len(src_toks), p+4)]),
                        'full_src': rec['text'],
                        'full_pred': pred
                    })
                    if len(sample_overcorrections) >= 20:
                        break

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    corr_acc = correct_at_tp / total_tp if total_tp > 0 else 0.0
    overcorr_rate = total_fp / total_clean_tokens if total_clean_tokens > 0 else 0.0
    clean_retention = clean_sents_preserved / clean_sents_total if clean_sents_total > 0 else 1.0

    return {
        'detection': {
            'tp': total_tp,
            'fp': total_fp,
            'fn': total_fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        },
        'correction': {
            'correct_at_tp': correct_at_tp,
            'accuracy': corr_acc,
        },
        'over_correction': {
            'fp_count': total_fp,
            'clean_tokens': total_clean_tokens,
            'rate': overcorr_rate,
            'clean_sents_total': clean_sents_total,
            'clean_sents_preserved': clean_sents_preserved,
            'clean_retention_rate': clean_retention,
        },
        'stratified': stratified,
        'sample_overcorrections': sample_overcorrections,
    }

# Sanity Test hàm đánh giá trên ví dụ giả định (copy nguyên vẹn từ nb3)
_test_recs = [{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}]
_test_preds = ['học sinh đi hoc'] # Model sửa 'sanh' -> 'sinh', nhưng bỏ sót 'hoc'
_m_test = evaluate_predictions(_test_recs, _test_preds)
assert _m_test['detection']['tp'] == 1 and _m_test['detection']['fn'] == 1 and _m_test['detection']['fp'] == 0
assert _m_test['correction']['accuracy'] == 1.0
print('Sanity check hàm evaluate_predictions PASS 100%!')

## 2. Dựng tập vị trí TP-detect-nhưng-sai-nội-dung

Giống semantics `evaluate_predictions` (nguồn CorrAcc nền 67,94%): gold/pred edit blocks align theo token nguồn; TP = vị trí có mặt ở cả 2 map; **sai-nội-dung** = TP mà `pred_str.lower() != gold_tgt.lower()`. Phân loại ghi vào từng vị trí: `deletion` nếu gold tgt rỗng (**loại khỏi oracle** — plan: không áp dụng cho deletion); còn lại `realword`/`nonword` theo **token lỗi phía src** (cùng convention stratified nb3/nb5: `src.lower() ∈ SYLL_SET` → realword) — khớp nhận định nb4 Q4 (~77% TP-sai là thay nhầm từ thật). Ghi thêm `n_tgt_tokens`: gold tgt đa token gần như chỉ có thể hit qua identity/confusion (MLM sinh theo token) — thống kê minh bạch.

In [ ]:
def load_prediction_set(path):
    recs = load_jsonl(path)
    preds = [r['prediction'] for r in recs]
    assert len(recs) == len(preds)
    for r in recs:
        assert 'text' in r and 'corrected_text' in r and 'prediction' in r, r.get('row_id')
    return recs, preds

noise_model = json.loads(Path(INPUT_FILES['noise_model.json']).read_text(encoding='utf-8'))
CONFUSION = noise_model['confusion']
ERROR_VOCAB = {e for errs in CONFUSION.values() for e in errs}
SYLL_TABLE = json.loads(Path(INPUT_FILES['syllable_table.json']).read_text(encoding='utf-8'))
SYLL_SET = set(SYLL_TABLE['entries'])
print(f'CONFUSION: {len(CONFUSION)} token sạch · ERROR_VOCAB: {len(ERROR_VOCAB)} · bảng âm tiết: {len(SYLL_SET)}')

val_recs, val_preds = load_prediction_set(INPUT_FILES['predictions_val_run2.jsonl'])
test_recs, test_preds = load_prediction_set(INPUT_FILES['predictions_test_run2.jsonl'])
_test_aligned = load_jsonl(INPUT_FILES['test_aligned.jsonl'])
assert len(test_recs) == len(_test_aligned), (len(test_recs), len(_test_aligned))
_mismatch = sum(1 for r, t in zip(test_recs, _test_aligned)
                if r['text'] != t['text'] or r['corrected_text'] != t['corrected_text'])
assert _mismatch == 0, f'test_run2: {_mismatch} dòng lệch text/corrected_text vs test_aligned.jsonl'
print('Đối chiếu test_run2 ↔ test_aligned.jsonl: KHỚP 100% (text + corrected_text, cùng thứ tự)')


def token_position_map(src_toks, other_toks):
    """Map vị trí token nguồn → (chuỗi đích của block, src_tokens) — chỉ block có token nguồn (như evaluate_predictions)."""
    blocks = extract_edit_blocks(src_toks, other_toks, levenshtein_opcodes(src_toks, other_toks))
    pos_map = {}
    for b in blocks:
        if b['src_span'][0] < b['src_span'][1]:
            target_str = ' '.join(b['tgt_tokens'])
            for p in range(b['src_span'][0], b['src_span'][1]):
                pos_map[p] = (target_str, b['src_tokens'])
    return pos_map


def wrong_content_positions(recs, preds, syll_set, dataset_name):
    """Vị trí TP-detect-nhưng-sai-nội-dung — cùng phép so .lower() với CorrAcc của evaluate_predictions."""
    rows = []
    for si, (rec, pred) in enumerate(zip(recs, preds)):
        src = canon_tokenize(rec['text'])
        gold_map = token_position_map(src, canon_tokenize(rec['corrected_text']))
        pred_map = token_position_map(src, canon_tokenize(pred))
        for p in sorted(set(gold_map) & set(pred_map)):       # TP-detect
            gold_tgt, _ = gold_map[p]
            pred_str, _ = pred_map[p]
            if pred_str.lower() == gold_tgt.lower():
                continue                                       # TP đúng nội dung
            src_low = nfc_normalize(src[p]).lower()
            rows.append({
                'dataset': dataset_name, 'sent_idx': si, 'row_id': rec.get('row_id'), 'position': p,
                'src_tok': src[p], 'gold_tgt': gold_tgt, 'pred_str': pred_str,
                'tgt_cat': 'deletion' if gold_tgt == '' else ('realword' if src_low in syll_set else 'nonword'),
                'n_tgt_tokens': len(gold_tgt.split()) if gold_tgt else 0,
                'context': ' '.join(src[max(0, p - 4):p]) + ' [ ] ' + ' '.join(src[p + 1:p + 5]),
                'src_tokens': src,
            })
    return rows


pos_test = wrong_content_positions(test_recs, test_preds, SYLL_SET, 'test')
pos_val = wrong_content_positions(val_recs, val_preds, SYLL_SET, 'val')

# CorrAcc nền tính LẠI cùng pipeline — verdict dùng số này
EVAL_TEST = evaluate_predictions(test_recs, test_preds, SYLL_SET)
EVAL_VAL = evaluate_predictions(val_recs, val_preds, SYLL_SET)

# --- Sanity nhân tạo (không đụng dữ liệu thật) ---
_r1 = wrong_content_positions([{'text': 'học sanh đi học', 'corrected_text': 'học sinh đi học'}],
                              ['học sịnh đi học'], SYLL_SET, 'sanity')
assert len(_r1) == 1 and _r1[0]['tgt_cat'] == 'nonword' and _r1[0]['gold_tgt'] == 'sinh' \
    and _r1[0]['pred_str'] == 'sịnh', _r1
_r2 = wrong_content_positions([{'text': 'học sanh đi hoc', 'corrected_text': 'học sinh đi học'}],
                              ['học sinh đi hoc'], SYLL_SET, 'sanity')
assert len(_r2) == 0, _r2       # TP đúng nội dung + FN → không có vị trí sai-nội-dung
_r3 = wrong_content_positions([{'text': 'học sinh đi', 'corrected_text': 'học đi'}],
                              ['học sịnh đi'], SYLL_SET, 'sanity')
assert len(_r3) == 1 and _r3[0]['tgt_cat'] == 'deletion', _r3
print('Sanity wrong-content PASS (sai nội dung · TP-đúng-nội-dung không tính · deletion)')

print()
print(f'Test: {len(pos_test)} vị trí sai-nội-dung / total_tp={EVAL_TEST["detection"]["tp"]} · '
      f'CorrAcc nền {EVAL_TEST["correction"]["accuracy"]:.2%} (tham chiếu REPORT §13: {BASE_CORRACC_TEST_REF:.2%}) · '
      f'{dict(collections.Counter(r["tgt_cat"] for r in pos_test))}')
print(f'Val : {len(pos_val)} vị trí sai-nội-dung / total_tp={EVAL_VAL["detection"]["tp"]} · '
      f'CorrAcc nền {EVAL_VAL["correction"]["accuracy"]:.2%} · '
      f'{dict(collections.Counter(r["tgt_cat"] for r in pos_val))}')
print(f'Gold tgt đa token (chỉ identity/confusion có thể hit): '
      f'test {sum(1 for r in pos_test if r["n_tgt_tokens"] > 1)} · val {sum(1 for r in pos_val if r["n_tgt_tokens"] > 1)}')

## 3. Candidate set — confusion (train-only) + MLM top-k (+ probe/fallback)

Thứ tự candidate pre-registered: `identity` → `confusion` (đảo `CONFUSION` nb2, xếp theo count giảm rồi alphabet — deterministic) → `MLM` (thứ tự logit/beam). Dedupe giữ vị trí đầu (khóa `.lower()`); gold rank = vị trí đầu tiên gold xuất hiện trong danh sách. MLM: BART/mBART đọc top-k logits decoder-step-0 (infill 1 token); T5/ViT5 beam-k rồi parse span `<extra_id_0>`. Probe smoke quyết định model dùng trước khi chạy thật; fail cả 2 → dừng với hướng dẫn (plan §rủi ro). Xong phần MLM giải phóng model ngay.

In [ ]:
# --- Đảo confusion (nguồn train-only, nb2) ---
CONF_REVERSE = {}
for _clean, _errs in CONFUSION.items():
    for _e in _errs:
        CONF_REVERSE.setdefault(_e, set()).add(_clean)


def confusion_candidates(src_low):
    """Token sạch từng bị gõ nhầm thành src_low — xếp (-count, token) cho deterministic."""
    return sorted(CONF_REVERSE.get(src_low, ()), key=lambda c: (-CONFUSION[c].get(src_low, 0), c))


# Sanity đảo confusion: mọi cặp (clean, error) của 500 token đầu phải đảo lại được
for _c, _es in list(CONFUSION.items())[:500]:
    for _e in _es:
        assert _c in CONF_REVERSE.get(_e, set()), (_c, _e)
print(f'CONF_REVERSE: {len(CONF_REVERSE)} dạng lỗi · sanity đảo 500 token đầu PASS')


# --- MLM infilling ---
def _clean_tok(s):
    return s.replace('▁', '').strip()


def mlm_topk_bart(model, tokenizer, token_lists, positions, k=TOP_K):
    """BART/mBART-style: thay token bằng <mask>, đọc top-k logits ở decoder step đầu (infill 1 token)."""
    mask = tokenizer.mask_token
    texts = [' '.join(toks[:p] + [mask] + toks[p + 1:]) for toks, p in zip(token_lists, positions)]
    outs = []
    for i in range(0, len(texts), MLM_BATCH_SIZE):
        enc = tokenizer(texts[i:i + MLM_BATCH_SIZE], padding=True, truncation=True,
                        max_length=MAX_SOURCE_LEN, return_tensors='pt').to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=1, num_beams=1,
                                 output_scores=True, return_dict_in_generate=True)
        scores = gen.scores[0]                                  # [batch, vocab] — logits token đầu ra
        for row_ids in scores.topk(k, dim=-1).indices.tolist():
            cands = []
            for tid in row_ids:
                if tid in tokenizer.all_special_ids:
                    continue
                t = _clean_tok(tokenizer.convert_ids_to_tokens([tid])[0])
                if t and t not in cands:
                    cands.append(t)
            outs.append(cands)
        if (i // MLM_BATCH_SIZE) % 20 == 0:
            print(f'  MLM {min(i + MLM_BATCH_SIZE, len(texts))}/{len(texts)}...')
    return outs


def mlm_topk_t5(model, tokenizer, token_lists, positions, k=TOP_K):
    """T5/ViT5 span-infilling: <mask>=<extra_id_0>, beam k, parse infill đầu tiên."""
    mask = tokenizer.mask_token or '<extra_id_0>'
    texts = [' '.join(toks[:p] + [mask] + toks[p + 1:]) for toks, p in zip(token_lists, positions)]
    outs = []
    for i in range(0, len(texts), MLM_BATCH_SIZE):
        enc = tokenizer(texts[i:i + MLM_BATCH_SIZE], padding=True, truncation=True,
                        max_length=MAX_SOURCE_LEN, return_tensors='pt').to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=8, num_beams=k, num_return_sequences=k,
                                 return_dict_in_generate=True)
        seqs = tokenizer.batch_decode(gen.sequences, skip_special_tokens=False)
        for bi in range(len(texts[i:i + MLM_BATCH_SIZE])):
            cands = []
            for seq in seqs[bi * k:(bi + 1) * k]:
                seg = seq.split(mask)[-1].split('<extra_id_1>')[0] if mask in seq else seq
                toks_gen = [t for t in _clean_tok(seg).split() if t]
                if toks_gen and toks_gen[0] not in cands:
                    cands.append(toks_gen[0])
            outs.append(cands[:k])
        if (i // MLM_BATCH_SIZE) % 20 == 0:
            print(f'  MLM {min(i + MLM_BATCH_SIZE, len(texts))}/{len(texts)}...')
    return outs


def load_mlm(model_id):
    print(f'Nạp MLM: {model_id} (device={DEVICE}, fp16={FP16})')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_id, torch_dtype=torch.float16 if FP16 else torch.float32).to(DEVICE)
    model.eval()
    return tokenizer, model


def run_probe(model, tokenizer, topk_fn):
    """Smoke probe: case hiển nhiên phải hit trong top-10 — chỉ phát hiện infill hỏng format, không phải benchmark."""
    mask = tokenizer.mask_token
    hits, detail = 0, []
    for tokens, expected in PROBE_CASES:
        toks = [mask if t == '<mask>' else t for t in tokens]
        cands = topk_fn(model, tokenizer, [toks], [toks.index(mask)], k=10)[0]
        ok = expected in [c.lower() for c in cands]
        hits += 1 if ok else 0
        detail.append({'tokens': tokens, 'expected': expected, 'hit': ok, 'top10': cands})
        print(f'  probe {"HIT" if ok else "miss"}: {" ".join(tokens)} (kỳ vọng {expected!r}) top10={cands}')
    return hits, detail


topk_fn, probe_detail = None, []
MLM_USED, MLM_PROBE_HITS = None, None
tokenizer, mlm_model = None, None
for _model_id, _fn in [(MLM_MODEL_ID, mlm_topk_bart), (MLM_FALLBACK_ID, mlm_topk_t5)]:
    tokenizer, mlm_model = load_mlm(_model_id)
    if tokenizer.mask_token is None:
        print(f'  {_model_id}: tokenizer thiếu mask token → thử fallback')
        del mlm_model
        continue
    _hits, probe_detail = run_probe(mlm_model, tokenizer, _fn)
    MLM_USED, MLM_PROBE_HITS, topk_fn = _model_id, _hits, _fn
    if _hits >= PROBE_MIN_HITS:
        break
    print(f'  {_model_id}: probe {_hits}/{len(PROBE_CASES)} < {PROBE_MIN_HITS} → fallback (ghi report)')
    del mlm_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    raise RuntimeError('Cả 2 MLM đều fail probe — kiểm tra Internet / mask format (plan §rủi ro: '
                       'bartpho-syllable infill kém → fallback ViT5 span-infilling).')
assert topk_fn is not None and MLM_USED is not None
print(f'>> Dùng MLM: {MLM_USED} · probe {MLM_PROBE_HITS}/{len(PROBE_CASES)} hit')

# --- Sinh MLM candidates cho toàn bộ vị trí sai-nội-dung (test + val, gộp batch) ---
_all_rows = pos_test + pos_val
print(f'Sinh MLM top-{TOP_K} cho {len(_all_rows)} vị trí (batch={MLM_BATCH_SIZE})...')
_mlm_lists = topk_fn(mlm_model, tokenizer, [r['src_tokens'] for r in _all_rows],
                     [r['position'] for r in _all_rows], k=TOP_K)
MLM_CANDS = {(_r['dataset'], _r['sent_idx'], _r['position']): _c for _r, _c in zip(_all_rows, _mlm_lists)}

del mlm_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('MLM candidates xong — đã giải phóng model.')

## 4. Đo oracle CorrAcc@k + breakdown + verdict

`oracle@k` = tỷ lệ vị trí sai-nội-dung (đã loại deletion) mà `gold_tgt` nằm trong k candidate đầu (so `.lower()` — nhất quán CorrAcc). **Projected CorrAcc oracle@k** = (correct_at_tp + hits@k) / total_tp — so với CorrAcc nền của **cùng bộ prediction** (tính lại ở §2). Verdict chỉ dùng test; val đọc kèm.

In [ ]:
def build_ranked_candidates(row):
    """Thứ tự pre-registered: identity → confusion → MLM. Dedupe giữ vị trí đầu (khóa .lower())."""
    ranked, seen = [], set()

    def _add(source, cand):
        key = cand.lower()
        if key and key not in seen:
            seen.add(key)
            ranked.append((source, cand))

    _add('identity', row['pred_str'])
    for c in confusion_candidates(nfc_normalize(row['src_tok']).lower()):
        _add('confusion', c)
    for c in MLM_CANDS.get((row['dataset'], row['sent_idx'], row['position']), []):
        _add('mlm', c)
    return ranked


def gold_hit(row):
    gold = row['gold_tgt'].lower()
    for rank, (source, cand) in enumerate(build_ranked_candidates(row), start=1):
        if cand.lower() == gold:
            return rank, source
    return None, None


for _r in pos_test + pos_val:
    if _r['tgt_cat'] != 'deletion':
        _r['gold_rank'], _r['gold_source'] = gold_hit(_r)


def oracle_table(rows, eval_m):
    """Oracle@k + projected CorrAcc + breakdown theo src-cat + nguồn hit."""
    total_tp = eval_m['detection']['tp']
    correct = eval_m['correction']['correct_at_tp']
    applicable = [r for r in rows if r['tgt_cat'] != 'deletion']
    table = {'n_wrong_content': len(rows),
             'n_excluded_deletion': len(rows) - len(applicable),
             'n_applicable': len(applicable),
             'n_tgt_multi_token': sum(1 for r in rows if r['n_tgt_tokens'] > 1),
             'k': {}}
    for k in K_ORACLE:
        hits = sum(1 for r in applicable if r['gold_rank'] is not None and r['gold_rank'] <= k)
        proj = (correct + hits) / total_tp if total_tp else 0.0
        table['k'][k] = {
            'hits': hits,
            'hit_rate': hits / len(applicable) if applicable else 0.0,
            'projected_corracc': proj,
            'corracc_gain_vs_base': proj - eval_m['correction']['accuracy'],
        }
    table['by_src_cat'] = {}
    for c in ('realword', 'nonword'):
        sub = [r for r in applicable if r['tgt_cat'] == c]
        table['by_src_cat'][c] = {
            'n': len(sub),
            'hits@10': sum(1 for r in sub if r['gold_rank'] is not None and r['gold_rank'] <= 10),
        }
    table['source_at_10'] = dict(collections.Counter(
        r['gold_source'] for r in applicable if r['gold_rank'] is not None and r['gold_rank'] <= 10))
    return table


oracle_results = {'test': oracle_table(pos_test, EVAL_TEST), 'val': oracle_table(pos_val, EVAL_VAL)}
for _name in ('test', 'val'):
    _t = oracle_results[_name]
    print(f'--- oracle · {_name} ---')
    print(f"  sai-nội-dung: {_t['n_wrong_content']} (loại deletion {_t['n_excluded_deletion']} · "
          f"áp dụng {_t['n_applicable']} · gold tgt đa token {_t['n_tgt_multi_token']})")
    for k in K_ORACLE:
        _kk = _t['k'][k]
        print(f"  @{k:<2d}: hit {_kk['hits']}/{_t['n_applicable']} ({_kk['hit_rate']:.2%}) → "
              f"projected CorrAcc {_kk['projected_corracc']:.2%} (Δ {_kk['corracc_gain_vs_base']:+.2%})")
    print(f"  theo src-cat (hits@10): {_t['by_src_cat']} · nguồn hit@10: {_t['source_at_10']}")

_t10 = oracle_results['test']['k'][ORACLE_K_DECISION]
BASE_CORRACC = EVAL_TEST['correction']['accuracy']
ORACLE_THRESHOLD = BASE_CORRACC + ORACLE_MIN_GAIN
ORACLE_PASS = _t10['projected_corracc'] >= ORACLE_THRESHOLD
print()
print('=== VERDICT PRE-REGISTERED (E0 — ngưỡng đã chốt ở §0) ===')
print(f'  Nền CorrAcc test (tính lại cùng pipeline): {BASE_CORRACC:.2%} '
      f'(tham chiếu REPORT §13: {BASE_CORRACC_TEST_REF:.2%})')
print(f"  Projected CorrAcc oracle@{ORACLE_K_DECISION} (test): {_t10['projected_corracc']:.2%} "
      f'→ cần >= {ORACLE_THRESHOLD:.2%}')
if ORACLE_PASS:
    print('>> ĐẠT NGƯỠNG: trần selection đủ lớn → MỞ E1 (rerank — plan riêng).')
else:
    print('>> KHÔNG ĐẠT: selection không có trần đủ lớn → DỪNG HƯỚNG B, ghi negative result vào REPORT.')

## 5. QA + xuất file

QA 10 vị trí sai-nội-dung đầu của test (ngữ cảnh, gold, pred, gold rank, top-10 candidate kèm nguồn). `mlm_oracle_report.json` gói: config + probe, oracle@k test/val, breakdown, verdict, positions chi tiết test (không kèm src_tokens cho gọn).

In [ ]:
qa = []
for r in pos_test:
    if r['tgt_cat'] == 'deletion' or len(qa) >= 10:
        continue
    qa.append({'row_id': r['row_id'], 'sent_idx': r['sent_idx'], 'position': r['position'],
               'context': r['context'], 'src_tok': r['src_tok'], 'gold_tgt': r['gold_tgt'],
               'pred_str': r['pred_str'], 'tgt_cat': r['tgt_cat'],
               'gold_rank': r.get('gold_rank'), 'gold_source': r.get('gold_source'),
               'top10': [{'source': s, 'cand': c} for s, c in build_ranked_candidates(r)[:10]]})
print('=== QA · test — 10 vị trí sai-nội-dung đầu ===')
for s in qa:
    print(f"[{s['row_id']} · pos {s['position']} · {s['tgt_cat']}] …{s['context']}…")
    print(f"  src={s['src_tok']!r} · gold={s['gold_tgt']!r} · pred={s['pred_str']!r} · "
          f"gold_rank={s['gold_rank']} ({s['gold_source']})")
    print('  top10:', [(x['source'], x['cand']) for x in s['top10']])
    print()

mlm_oracle_report = {
    'created': RUN_STAMP,
    'notebook': 'nb9_mlm_rerank_oracle',
    'shared_cells_version': SHARED_CELLS_VERSION,
    'seed': SEED,
    'pre_registered': {
        'oracle_k_decision': ORACLE_K_DECISION,
        'min_corracc_gain': ORACLE_MIN_GAIN,
        'base_corracc_test_reference': BASE_CORRACC_TEST_REF,
        'candidate_order': 'identity → confusion (đảo CONFUSION nb2: -count rồi alphabet) → MLM top-k',
    },
    'mlm': {'model_used': MLM_USED, 'fallback_id': MLM_FALLBACK_ID,
            'probe_hits': MLM_PROBE_HITS, 'probe_cases': len(PROBE_CASES), 'probe_detail': probe_detail},
    'leakage_note': 'Candidates chỉ từ MLM công khai + CONFUSION train (nb2); nhãn test chỉ dùng để đo hit (DESIGN.md §9).',
    'baseline': {
        'test': {'corracc': BASE_CORRACC, 'total_tp': EVAL_TEST['detection']['tp'],
                 'correct_at_tp': EVAL_TEST['correction']['correct_at_tp']},
        'val': {'corracc': EVAL_VAL['correction']['accuracy']},
    },
    'oracle': oracle_results,
    'verdict_e0': {'projected_corracc_test': _t10['projected_corracc'],
                   'threshold': ORACLE_THRESHOLD, 'pass': ORACLE_PASS,
                   'action': 'mở E1 (plan riêng)' if ORACLE_PASS else 'dừng Hướng B — negative result vào REPORT'},
    'qa_samples_test': qa,
    'positions_test': [{k: v for k, v in r.items() if k != 'src_tokens'} for r in pos_test],
    'files': ['mlm_oracle_report.json'],
}
with open(OUTPUT_DIR / 'mlm_oracle_report.json', 'w', encoding='utf-8') as f:
    json.dump(mlm_oracle_report, f, ensure_ascii=False, indent=2)

print('=== HOÀN TẤT nb9 (Hướng B · E0) ===')
print('Đã ghi vào', OUTPUT_DIR)
for fn in mlm_oracle_report['files']:
    print(' -', fn)